# Ćwiczenia 7 — Ocena modelu: macierz pomyłek, krzywa ROC, próg (wersja Python)
**Dane przykładowe: kohorta T2D (QinJ_2012)**

Sama dokładność nie mówi, jakie błędy model popełnia, i zależy od przyjętego progu. Na tym spotkaniu oceniamy dwa znane już modele, regresję logistyczną i k-NN, dokładniej: liczymy macierz pomyłek i miary z niej, krzywą ROC z polem pod nią (AUC) i sprawdzamy, jak zmiana progu przesuwa czułość i swoistość. Nie budujemy nowych modeli.

Pracujemy na przewidywaniach z walidacji krzyżowej na zbiorze treningowym (Ćwiczenia 5). Zbiór testowy odkładamy do oceny końcowej na Ćwiczeniach 9. Notatnik najlepiej uruchamiać od początku po ponownym uruchomieniu jądra (*Kernel → Restart*), komórka po komórce. Liczby różnią się nieco od wersji R, bo oba języki inaczej losują podział. Wpływ progu na czułość i swoistość jest taki sam, ale różnice między modelami są małe, więc w Zadaniach 1 i 2 lepszy może okazać się inny model niż w wersji R.

## Punkt wyjścia

Przygotowanie danych i części walidacji jak na Ćwiczeniach 6.

In [ ]:
import numpy as np                       # obliczenia na tablicach
import pandas as pd                      # tabele danych
import matplotlib.pyplot as plt          # wykresy
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict   # podział, części, przewidywania z walidacji
from sklearn.pipeline import make_pipeline             # potok: kroki i model w jednym obiekcie
from sklearn.preprocessing import StandardScaler       # standaryzacja
from sklearn.decomposition import PCA                  # analiza głównych składowych
from sklearn.neighbors import KNeighborsClassifier     # k najbliższych sąsiadów
from sklearn.linear_model import LogisticRegression    # regresja logistyczna
from sklearn.metrics import confusion_matrix, recall_score, precision_score, f1_score, balanced_accuracy_score   # miary
from sklearn.metrics import roc_auc_score, roc_curve, RocCurveDisplay   # krzywa ROC i pole pod nią
pd.set_option("display.width", 120)      # szersze wydruki tabel

X = pd.read_csv("dane/t2d_features.csv", index_col=0)   # index_col=0: pierwsza kolumna to nazwy próbek
# ["grupa"]: kolumna z etykietami; index=X.index: te same nazwy wierszy co w X
y = pd.Series(pd.read_csv("dane/t2d_labels.csv")["grupa"].to_numpy(), index=X.index)
# test_size=0.25: 25% próbek do testu; stratify=y: osobno w każdej klasie; random_state=42: ziarno kursu
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

prevalence_tr = (X_tr > 0).mean()        # częstość występowania każdej cechy w treningu
keep = prevalence_tr.index[prevalence_tr >= 0.10]   # nazwy cech obecnych w co najmniej 10% próbek

# CLR: logarytm (z dodaną stałą 1e-6, bo log(0) nie istnieje) minus średnia logarytmów w próbce
def clr(M):
    L = np.log(M + 1e-6)                 # logarytm każdej wartości
    return L.sub(L.mean(axis=1), axis=0) # od każdego wiersza (próbki) odejmujemy jego średnią

Z_tr = clr(X_tr[keep])                   # cechy treningowe po filtrze i CLR
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)   # te same 5 części co na Ćwiczeniach 6

Oba modele to te same potoki co na Ćwiczeniach 6. Nowa jest funkcja `cross_val_predict()`: zamiast dokładności w każdej części zwraca dla każdej próbki treningowej przewidywanie modelu, którego na niej nie uczono (bo była wtedy w części sprawdzającej). Z `method="predict_proba"` są to prawdopodobieństwa klas. Na tych prawdopodobieństwach liczymy wszystkie miary.

In [ ]:
lr = make_pipeline(StandardScaler(), PCA(n_components=10), LogisticRegression(C=np.inf))     # regresja logistyczna
knn = make_pipeline(StandardScaler(), PCA(n_components=10), KNeighborsClassifier(n_neighbors=15))   # k-NN, k = 15

# prawdopodobieństwa z walidacji krzyżowej; kolumny w kolejności alfabetycznej klas (wielkie litery przed małymi): [:, 0] to "T2D"
p_lr = cross_val_predict(lr, Z_tr, y_tr, cv=cv, method="predict_proba")[:, 0]
p_knn = cross_val_predict(knn, Z_tr, y_tr, cv=cv, method="predict_proba")[:, 0]   # to samo dla k-NN
obs = y_tr.to_numpy()                    # prawdziwe klasy
print(pd.DataFrame({"obs": obs[:6], "P_T2D": p_lr[:6].round(3)}))   # pierwsze 6 próbek

## 1. Macierz pomyłek i miary przy jednym progu

Dokładność łączy wszystkie błędy w jedną liczbę, a różne błędy mają różne skutki: pominięcie osoby chorej to co innego niż fałszywy alarm u osoby zdrowej. Macierz pomyłek (ang. *confusion matrix*) pokazuje oba rodzaje błędów osobno. T2D jest klasą pozytywną (ang. *positive class*), czyli tą, którą chcemy wykryć. Z macierzy liczy się:

- **czułość** (*sensitivity*, w scikit-learn *recall*) — odsetek próbek T2D rozpoznanych jako T2D,
- **swoistość** (*specificity*) — odsetek próbek control rozpoznanych jako control (czułość liczona dla klasy control),
- **precyzja** (*precision*) — odsetek próbek T2D wśród wszystkich wyników dodatnich,
- **F1** — łączy precyzję i czułość; jest wysoka tylko wtedy, gdy obie są wysokie.

Prawdopodobieństwo zamieniamy na klasę progiem 0,5.

In [ ]:
threshold = 0.5                          # próg: P(T2D) ≥ próg oznacza T2D
pred = np.where(p_lr >= threshold, "T2D", "control")   # klasa przy tym progu
# labels: kolejność wierszy i kolumn; wiersze = prawdziwa klasa, kolumny = przewidywanie (odwrotnie niż w caret)
print(pd.DataFrame(confusion_matrix(obs, pred, labels=["control", "T2D"]),
                   index=["true control", "true T2D"], columns=["pred control", "pred T2D"]))   # podpisy wierszy i kolumn
print({"sensitivity": recall_score(obs, pred, pos_label="T2D"),        # czułość
       "specificity": recall_score(obs, pred, pos_label="control"),    # swoistość
       "precision": precision_score(obs, pred, pos_label="T2D"),       # precyzja
       "F1": f1_score(obs, pred, pos_label="T2D"),                     # F1
       "balanced_accuracy": balanced_accuracy_score(obs, pred)})       # dokładność zbalansowana

**Na co patrzeć.** Na przekątnej macierzy leżą poprawne klasyfikacje. Wiersz `true control` w kolumnie `pred T2D` to fałszywe alarmy, wiersz `true T2D` w kolumnie `pred control` to pominięte próbki T2D. Dokładność zbalansowana to średnia czułości i swoistości; przyda się w sekcji 4.

> **✏️ Zadanie 1.** Policzyć macierz pomyłek i miary dla k-NN (te same linijki, z `p_knn` zamiast `p_lr`). Który model popełnia więcej fałszywych alarmów, a który częściej pomija próbki T2D?

In [ ]:
# miejsce na kod do Zadania 1

## 2. Krzywa ROC i AUC

Miary z sekcji 1 zależą od progu. Żeby ocenić sam model, sprawdza się wszystkie progi jednocześnie. **Krzywa ROC** pokazuje czułość względem odsetka fałszywych alarmów (1 − swoistość) dla kolejnych progów. **AUC** (pole pod krzywą ROC) to prawdopodobieństwo, że model przypisze losowo wybranej próbce T2D wyższe prawdopodobieństwo T2D niż losowo wybranej próbce control: 0,5 oznacza model losowy, 1 idealne uporządkowanie próbek.

In [ ]:
print(roc_auc_score(obs == "T2D", p_lr))   # pole pod krzywą ROC; obs == "T2D": klasa pozytywna jako True
RocCurveDisplay.from_predictions(obs, p_lr, pos_label="T2D", plot_chance_level=True)   # krzywa ROC z przekątną
plt.show()                               # wyświetlenie wykresu

**Na co patrzeć.** Im bliżej lewego górnego rogu leży krzywa, tym lepiej model porządkuje próbki. Przekątna to poziom modelu losowego.

Gdy klasa pozytywna jest rzadka, uzupełnieniem krzywej ROC jest krzywa precyzja–czułość (PR); w tej kohorcie klasy są prawie równoliczne, więc wystarcza krzywa ROC.

> **✏️ Zadanie 2.** Policzyć AUC dla k-NN (`roc_auc_score()` z `p_knn`). Który model lepiej porządkuje próbki? O ile AUC obu modeli przekracza 0,5 (poziom modelu losowego) i czy różnica między modelami jest duża w porównaniu z tą przewagą?

In [ ]:
# miejsce na kod do Zadania 2

## 3. Wybór progu

Próg 0,5 jest umowny. Obniżenie progu zwiększa czułość kosztem swoistości, podwyższenie działa odwrotnie, więc próg dobiera się do celu: w badaniu przesiewowym ważniejsze jest, żeby nie pominąć chorych, a przed kosztownym leczeniem, żeby nie leczyć zdrowych. Pętla poniżej liczy czułość i swoistość przy trzech progach. Jedną z reguł wyboru progu jest **indeks Youdena**: wybiera się próg, przy którym suma czułości i swoistości jest największa.

In [ ]:
for t in [0.3, 0.5, 0.7]:                # dla każdego z trzech progów wypisuje: próg, czułość, swoistość
    pred_t = np.where(p_lr >= t, "T2D", "control")   # klasa przy progu t
    print(t, round(recall_score(obs, pred_t, pos_label="T2D"), 3),     # czułość
          round(recall_score(obs, pred_t, pos_label="control"), 3))    # swoistość

fpr, tpr, thr = roc_curve(obs, p_lr, pos_label="T2D")   # punkty krzywej ROC: 1 − swoistość, czułość, próg
best = np.argmax(tpr - fpr)              # punkt o największej sumie czułości i swoistości (indeks Youdena)
print(thr[best], tpr[best], 1 - fpr[best])   # próg Youdena, czułość i swoistość przy nim

**Na co patrzeć.** Im wyższy próg, tym niższa czułość i wyższa swoistość. Próg wybrany na tych samych danych, na których się go ocenia, daje wynik zbyt optymistyczny, dlatego wybiera się go na treningu (tu: z walidacji krzyżowej), a sprawdza na zbiorze testowym.

> **✏️ Zadanie 3.** Policzyć macierz pomyłek regresji logistycznej przy progu 0,3 (te same linijki co w sekcji 1, z `threshold = 0.3`). Ile próbek T2D model teraz pomija, a ile było ich przy progu 0,5? O ile wzrasta przy tym liczba fałszywych alarmów?

In [ ]:
# miejsce na kod do Zadania 3

## 4. Niezrównoważenie klas

W kohorcie demonstracyjnej klasy są prawie równoliczne, ale w wielu zbiorach (także w części kohort projektowych) jedna klasa jest wyraźnie rzadsza. Wtedy model, który zawsze wskazuje klasę liczniejszą, ma wysoką dokładność, choć jest bezużyteczny: przy 10% próbek T2D jego dokładność wynosi 0,90, a czułość 0. Dlatego przy niezrównoważonych klasach:

- wynik porównuje się z klasyfikatorem bazowym, a obok dokładności podaje czułość, swoistość i dokładność zbalansowaną (`balanced_accuracy_score()`);
- podział na zbiór treningowy, testowy i części walidacji robi się warstwowo (`stratify=y`, `StratifiedKFold`);
- próg dobiera się do celu (sekcja 3) albo nadaje próbkom rzadszej klasy większą wagę podczas uczenia (`LogisticRegression(class_weight="balanced")`; w caret argument `weights` w `train()`).